# Stage 3 — Preprocessing & Class Balancing

**Thesis:** Predictive Analytics for MSME Credit Risk Assessment using Behavioural Feature Engineering and Explainable Ensemble Machine Learning

Notebook 3 of 5. Takes `outputs/features.parquet` from Notebook 2 and turns it
into something I can actually model with, without leaking test information
anywhere:

1. Type the columns properly and drop anything degenerate
2. **Stratified 80/20 train-test split** (fixed seed) - saved to disk so
   Notebooks 4–5 use exactly the same rows
3. Build the preprocessing recipe: **median** imputation (numeric) /
   **most-frequent** imputation (categorical), **one-hot** encoding, **min-max**
   normalisation
4. **SMOTE** to balance the classes - shown here on the training partition, and
   re-fitted inside every CV fold in Notebook 4 so it never touches validation data
5. A few leakage checks plus a before/after picture

Outputs: `outputs/split_train.parquet`, `outputs/split_test.parquet`,
`outputs/preprocessor.joblib`.


In [1]:

import os, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
plt.ioff()
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
OUT_DIR = os.path.join(ROOT, "outputs")

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.family": "DejaVu Sans", "font.size": 11,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.labelsize": 11,
    "axes.edgecolor": "#333333", "axes.linewidth": 0.8,
})
C_REPAID, C_DEFAULT = "#4C72B0", "#C44E52"

def savefig(fig, name, caption=""):
    path = os.path.join(OUT_DIR, name)
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"saved -> outputs/{name}" + (f"   |  {caption}" if caption else ""))

df = pd.read_parquet(os.path.join(OUT_DIR, "features.parquet"))
print("features.parquet:", df.shape)
df.head(3)


features.parquet: (38412, 86)


,SK_ID_CURR,TARGET,repay_consistency__n_instalments,repay_consistency__ontime_ratio,repay_consistency__late_ratio,repay_consistency__dpd30_ratio,repay_consistency__days_late_mean,repay_consistency__days_late_max,repay_consistency__days_late_std,payment_shortfall__short_ratio,...,raw__credit_income_ratio,raw__annuity_income_ratio,raw__credit_goods_ratio,raw__credit_term,has__repay_consistency,has__payment_shortfall,has__delinquency,has__credit_utilisation,has__bureau_depth,has__prev_app
0,100017,0,30.0,1.0,0.0,0.0,-10.866667,-3.0,4.875932,0.0,...,4.082080,0.128740,1.3168,31.707938,1,1,1,0,1,1
1,100024,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.166667,0.158333,1.0000,20.000000,0,0,0,0,0,0
2,100026,0,8.0,1.0,0.0,0.0,-9.375000,-2.0,4.340425,0.0,...,1.105600,0.072270,1.1056,15.298187,1,1,1,0,1,1


## 1. Column typing and cleaning

- `SK_ID_CURR` is just an identifier, not a feature - drop it from `X`, keep it as
  the index.
- A few ratio features had a zero denominator slip through as `±inf` - set those
  to missing instead.
- Any numeric column with **zero variance** isn't telling the model anything, so
  drop it.
- Split what's left into numeric vs categorical, which is what drives the
  `ColumnTransformer` below.


In [2]:
df = df.set_index("SK_ID_CURR")
y = df.pop("TARGET").astype("int8")
X = df.replace([np.inf, -np.inf], np.nan)

cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X.select_dtypes(include="number").columns.tolist()

# drop zero-variance numeric columns
zv = [c for c in num_cols if X[c].nunique(dropna=True) <= 1]
if zv:
    X = X.drop(columns=zv)
    num_cols = [c for c in num_cols if c not in zv]
    print("dropped zero-variance columns:", zv)

print(f"\nfeatures: {X.shape[1]}   ({len(num_cols)} numeric, {len(cat_cols)} categorical)")
print("categorical:", cat_cols)
print("\nmissingness summary (numeric):")
print(X[num_cols].isna().mean().describe().round(3).to_string())



features: 84   (76 numeric, 8 categorical)
categorical: ['raw__NAME_CONTRACT_TYPE', 'raw__CODE_GENDER', 'raw__FLAG_OWN_CAR', 'raw__FLAG_OWN_REALTY', 'raw__NAME_INCOME_TYPE', 'raw__NAME_EDUCATION_TYPE', 'raw__NAME_FAMILY_STATUS', 'raw__NAME_HOUSING_TYPE']

missingness summary (numeric):
count    76.000
mean      0.164
std       0.255
min       0.000
25%       0.000
50%       0.033
75%       0.178
max       0.778


## 2. Stratified 80/20 train–test split

Stratified on `TARGET` so both partitions keep the same ~10.2% default rate, with
a fixed seed so it's reproducible. I write both partitions to disk **before** any
imputation, encoding or balancing happens - every notebook after this one loads
these exact rows, and the test partition doesn't get touched again until final
evaluation.


In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)

split = pd.DataFrame({
    "partition": ["train", "test"],
    "n":         [len(y_train), len(y_test)],
    "n_default": [int(y_train.sum()), int(y_test.sum())],
    "default_rate": [y_train.mean(), y_test.mean()],
})
print(split.round(4).to_string(index=False))

# save the raw (un-transformed) split so later notebooks all work from the same rows
train_out = X_train.copy(); train_out["TARGET"] = y_train
test_out  = X_test.copy();  test_out["TARGET"]  = y_test
train_out.reset_index().to_parquet(os.path.join(OUT_DIR, "split_train.parquet"), index=False)
test_out.reset_index().to_parquet(os.path.join(OUT_DIR, "split_test.parquet"), index=False)
print("\nsaved -> outputs/split_train.parquet, outputs/split_test.parquet")

assert abs(y_train.mean() - y_test.mean()) < 0.005, "stratification failed"
print("leakage check 1: train and test default rates match (stratified) — OK")


partition     n  n_default  default_rate
    train 30729       3126        0.1017
     test  7683        782        0.1018



saved -> outputs/split_train.parquet, outputs/split_test.parquet
leakage check 1: train and test default rates match (stratified) — OK


## 3. Preprocessing recipe

One `ColumnTransformer`:

| Block | Steps |
|---|---|
| Numeric | median imputation → min–max scaling to [0, 1] |
| Categorical | most-frequent imputation → one-hot encoding (levels with <20 rows grouped together) |

It's **fitted on the training partition only**. I save the fitted version to
`outputs/preprocessor.joblib`, but Notebook 4 actually rebuilds an unfitted copy
of the same recipe inside its own cross-validation pipeline, so the imputation
values and scaler ranges get learned fresh in every fold rather than reused from
here.


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

def build_preprocessor(num_cols, cat_cols):
    numeric = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale",  MinMaxScaler()),
    ])
    categorical = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20, sparse_output=False)),
    ])
    return ColumnTransformer([
        ("num", numeric, num_cols),
        ("cat", categorical, cat_cols),
    ], remainder="drop", verbose_feature_names_out=False)

preprocessor = build_preprocessor(num_cols, cat_cols)
Xtr = preprocessor.fit_transform(X_train, y_train)
Xte = preprocessor.transform(X_test)
feat_names = preprocessor.get_feature_names_out()

joblib.dump({"preprocessor": preprocessor, "num_cols": num_cols, "cat_cols": cat_cols,
             "feature_names": list(feat_names), "random_state": RANDOM_STATE},
            os.path.join(OUT_DIR, "preprocessor.joblib"))

print(f"transformed train: {Xtr.shape}   test: {Xte.shape}")
print(f"feature count after one-hot: {len(feat_names)}")
print("range check (should be within [0,1] for numeric):",
      f"min={Xtr.min():.3f}, max={Xtr.max():.3f}")
print("NaNs remaining:", int(np.isnan(Xtr).sum()), "(train)", int(np.isnan(Xte).sum()), "(test)")
print("\nsaved -> outputs/preprocessor.joblib")


transformed train: (30729, 104)   test: (7683, 104)
feature count after one-hot: 104
range check (should be within [0,1] for numeric): min=0.000, max=1.000
NaNs remaining: 0 (train) 0 (test)

saved -> outputs/preprocessor.joblib


## 4. SMOTE class balancing (training partition only)

SMOTE (from `imbalanced-learn`) creates new synthetic minority-class (default)
rows by interpolating between each minority point and its nearest minority
neighbours, until the training set reaches a 1:1 class ratio. It only ever runs
**after** preprocessing and **only** on the training partition - the test set
keeps its natural 10.2% default rate throughout.

For the actual models in Notebook 4, SMOTE lives inside an `imblearn.Pipeline` so
it gets refit within every CV fold. What's here is just for inspection and the
before/after picture.


In [5]:
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
Xtr_bal, ytr_bal = sm.fit_resample(Xtr, y_train)

before = pd.Series(y_train).value_counts().sort_index()
after  = pd.Series(ytr_bal).value_counts().sort_index()
print("Training set class counts")
print(f"  before SMOTE : repaid {before[0]:,}  |  default {before[1]:,}  "
      f"({before[1]/before.sum()*100:.1f}% default)")
print(f"  after  SMOTE : repaid {after[0]:,}  |  default {after[1]:,}  "
      f"({after[1]/after.sum()*100:.1f}% default)")
print(f"  synthetic default rows created: {after[1] - before[1]:,}")

test_counts = pd.Series(y_test).value_counts().sort_index()
print(f"\nTest set unchanged: repaid {test_counts[0]:,} | default {test_counts[1]:,} "
      f"({y_test.mean()*100:.1f}% default)")
print("leakage check 2: SMOTE applied to training rows only, test rate still natural — OK")


Training set class counts
  before SMOTE : repaid 27,603  |  default 3,126  (10.2% default)
  after  SMOTE : repaid 27,603  |  default 27,603  (50.0% default)
  synthetic default rows created: 24,477

Test set unchanged: repaid 6,901 | default 782 (10.2% default)
leakage check 2: SMOTE applied to training rows only, test rate still natural — OK


In [6]:
# figure: class balance before/after SMOTE, plus a 2-D PCA view of the synthetic points
from sklearn.decomposition import PCA

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))

# (a) bar chart
w = 0.35
axes[0].bar([0 - w/2, 1 - w/2], before.values, w, label="before SMOTE",
            color=["#9DB4C8", "#E0A0A2"], edgecolor="black", linewidth=0.4)
axes[0].bar([0 + w/2, 1 + w/2], after.values, w, label="after SMOTE",
            color=[C_REPAID, C_DEFAULT], edgecolor="black", linewidth=0.4)
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(["Repaid (0)", "Default (1)"])
axes[0].set_ylabel("Training rows")
axes[0].set_title("Training-set class balance before and after SMOTE")
for i, (b, a) in enumerate(zip(before.values, after.values)):
    axes[0].text(i - w/2, b + 400, f"{b:,}", ha="center", fontsize=8)
    axes[0].text(i + w/2, a + 400, f"{a:,}", ha="center", fontsize=8)
axes[0].legend()
sns.despine(ax=axes[0])

# (b) PCA scatter of just the minority class - real points vs synthetic ones
pca = PCA(n_components=2, random_state=RANDOM_STATE).fit(Xtr)
n_orig = len(Xtr)
real_min = pca.transform(Xtr[y_train.values == 1])
synth_min = pca.transform(Xtr_bal[n_orig:])
rng = np.random.default_rng(RANDOM_STATE)
ss = rng.choice(len(synth_min), 4000, replace=False)
axes[1].scatter(synth_min[ss, 0], synth_min[ss, 1], s=7, alpha=0.20, color="#E8B84B",
                label=f"synthetic default (n={len(synth_min):,})")
axes[1].scatter(real_min[:, 0], real_min[:, 1], s=9, alpha=0.55, color=C_DEFAULT,
                label=f"real default (n={len(real_min):,})")
axes[1].set_xlabel("Principal component 1"); axes[1].set_ylabel("Principal component 2")
axes[1].set_title("SMOTE synthetic vs real default cases (PCA projection)")
axes[1].legend(markerscale=2, fontsize=8, loc="lower right")
sns.despine(ax=axes[1])

fig.tight_layout()
savefig(fig, "prep_01_smote_balance.png",
        "Training-set class balance before and after SMOTE (left) and a 2-D PCA "
        "projection of the default class (right), showing that the synthetic examples "
        "occupy the same region as the real minority cases. The test partition is not resampled.")


saved -> outputs/prep_01_smote_balance.png   |  Training-set class balance before and after SMOTE (left) and a 2-D PCA projection of the default class (right), showing that the synthetic examples occupy the same region as the real minority cases. The test partition is not resampled.


## 5. Stage 3 summary

- **Split:** stratified 80/20, seed 42 → `outputs/split_train.parquet` (30,729
  rows), `outputs/split_test.parquet` (7,683 rows); both keep the 10.2% default rate.
- **Preprocessing:** median/most-frequent imputation → one-hot → min-max, fitted
  on the training partition only, saved to `outputs/preprocessor.joblib`.
- **SMOTE:** training partition rebalanced to 1:1, test partition left alone.
- **Leakage checks:** both passed - (1) stratification holds, (2) SMOTE and every
  `fit`/`fit_transform` call only ever saw training rows.
- Figure `prep_01` - class balance and PCA view of the synthetic points.

Next up (Notebook 4): `imblearn.Pipeline([preprocessor, SMOTE, model])` inside
`GridSearchCV` for Logistic Regression, Random Forest and XGBoost, evaluated on
the held-out test partition with default-class recall as the metric that matters.
